# refractivesqlite — Tutorial

A Python 3 + SQLite wrapper for the [refractiveindex.info](http://refractiveindex.info/) database of optical constants by [Mikhail Polyanskiy](https://github.com/polyanskiy).  
Package by [Hugo Guillén](https://github.com/HugoGuillen).

## Features
- Create a local SQLite database from the refractiveindex YML folder or from the upstream `.zip` URL.
- Search materials by name, shelf/book/page, refractive index (*n*), or extinction coefficient (*k*).
- Execute raw SQL queries for power users.
- Export data to NumPy arrays or CSV files.
- Retrieve *n*, *k*, and complex permittivity ε at arbitrary wavelengths — **scalar or array input**.
- Query using **any wavelength unit**: nm, µm, m, mm, Å, cm⁻¹, THz, or eV.
- Out-of-range wavelengths return **NaN** for clean vectorised workflows.

## Database schema

![Schema](../docs/ER.PNG "Schema")

| Table | Key columns |
|---|---|
| `pages` | `pageid`, `shelf`, `book`, `page`, `filepath`, `hasrefractive`, `hasextinction`, `rangeMin`, `rangeMax`, `points` |
| `refractiveindex` | `pageid`, `wave` (µm), `refindex` |
| `extcoeff` | `pageid`, `wave` (µm), `coeff` |

## Package structure

After the refactor the package is organised into focused single-responsibility modules:

```
refractivesqlite/
├── __init__.py        # public API: Database, Material
├── database.py        # Database class — search & retrieval
├── material.py        # Material class — per-material interface
├── optical_data.py    # Formula/tabulated n and k data classes
├── builder.py         # DB creation from YML folder (with catalog cache)
├── downloader.py      # Download upstream .zip
├── models.py          # Shelf / Book / Page / Entry namedtuples
├── exceptions.py      # NoExtinctionCoefficient, FormulaNotImplemented
├── _constants.py      # RII_DATABASE_URL
└── _units.py          # Wavelength unit registry (to_nm / from_nm)
```

The public API you need day-to-day is just two names:

```python
from refractivesqlite import Database, Material
```

---
## 1. Create / open the database

> **Once you have created the database you do not need to recreate it.** Skip straight to *1.4 Open an existing database* on subsequent runs.

### 1.1 Create from the upstream URL

Downloads the default database zip and populates an SQLite file.  
You can specify `interpolation_points` (default 100) to control how densely formula-based materials are sampled.

In [ ]:
from refractivesqlite import Database

DB_PATH = "refractive.db"
db = Database(DB_PATH)
db.create_database_from_url(interpolation_points=100)

### 1.2 Create from a custom URL

Pass any historical `.zip` URL from [refractiveindex.info/download](http://refractiveindex.info/download.php).

In [ ]:
from refractivesqlite import Database

DB_PATH = "refractive.db"
db = Database(DB_PATH)
db.create_database_from_url(
    riiurl="https://refractiveindex.info/download/database/rii-database-2025-02-23.zip"
)

### 1.3 Create from a local YML folder

If you have already downloaded and unzipped the database, point directly at the folder.

In [ ]:
from refractivesqlite import Database

DB_PATH = "refractive.db"
YML_PATH = "database"          # folder that contains library.yml
db = Database(DB_PATH)
db.create_database_from_folder(YML_PATH, interpolation_points=200)

### 1.4 Open an existing database

This is the normal entry-point on every run after the first.

In [ ]:
from refractivesqlite import Database

DB_PATH = "refractive.db"
db = Database(DB_PATH)

### 1.5 Check the default download URL

In [ ]:
db.check_url_version()

---
## 2. Searching the database

### 2.1 List all pages

Calling `search_pages()` with no arguments returns every entry in the database.

In [ ]:
db.search_pages()

### 2.2 Search by term (fuzzy)

The term is matched against `shelf`, `book`, `page`, and `filepath` with `LIKE %term%`.

In [ ]:
db.search_pages("otanicar")

### 2.3 Search by term (exact)

Set `exact=True` to require a case-insensitive exact match.

In [ ]:
db.search_pages("au", exact=True)

### 2.4 Search by page ID

Once you know a `pageid` you can look up its metadata directly.

In [ ]:
db.search_id(1542)

### 2.5 Search by refractive index (*n*) interval

Find all materials where *n* falls in `[n - delta_n, n + delta_n]`.

In [ ]:
db.search_n(n=0.3, delta_n=0.0001)

### 2.6 Search by extinction coefficient (*k*) interval

Find all materials where *k* falls in `[k - delta_k, k + delta_k]`.

In [ ]:
db.search_k(k=0.3, delta_k=0.0001)

### 2.7 Search by *n* and *k* simultaneously

Only returns rows where *both* constraints are met at the same wavelength.

In [ ]:
db.search_nk(n=0.3, delta_n=0.1, k=0.3, delta_k=0.1)

### 2.8 Custom SQL queries

`search_custom` runs any SQL string and returns a list of tuples — one per row.  
Refer to the schema diagram at the top for column names.

In [ ]:
# All pages in main/Ag whose name contains 'k'
results = db.search_custom(
    'SELECT * FROM pages WHERE shelf="main" AND book="Ag" AND page LIKE "%k%"'
)
print(results)

In [ ]:
# Extinction coefficients for page 1 in wavelength range [0.3, 0.4] µm
results = db.search_custom(
    "SELECT wave, coeff FROM extcoeff WHERE pageid = 1 AND wave BETWEEN 0.3 AND 0.4"
)
print(results)

In [ ]:
# All materials with both n and k measured at exactly 0.301 µm
results = db.search_custom("""
    SELECT p.filepath, r.wave, r.refindex, e.coeff
    FROM refractiveindex r
    INNER JOIN extcoeff e ON r.pageid = e.pageid AND r.wave = e.wave
    INNER JOIN pages p ON r.pageid = p.pageid
    WHERE r.wave = 0.301
""")
for row in results:
    print(row)

---
## 3. Working with a `Material` object

`db.get_material(pageid)` returns a `Material` instance that wraps the optical data for one entry.  
All subsequent operations on that entry go through this object.

### 3.1 Load a material

In [ ]:
# Page 5 = main / Ag / Johnson
mat = db.get_material(5)

### 3.2 Inspect metadata

In [ ]:
info = mat.get_page_info()
print(info)

print(f"\nShelf : {info['shelf']}")
print(f"Book  : {info['book']}")
print(f"Page  : {info['page']}")
print(f"Range : {info['rangeMin']} – {info['rangeMax']} µm")
print(f"Points: {info['points']}")

### 3.3 Check data availability

In [ ]:
print("Has refractive index (n)?", mat.has_refractive())
print("Has extinction coefficient (k)?", mat.has_extinction())

### 3.4 Get *n* and *k* as NumPy arrays

Each array has shape `(N, 2)`: column 0 is wavelength in µm, column 1 is the value.

In [ ]:
import numpy as np

n_data = np.array(mat.get_complete_refractive())
k_data = np.array(mat.get_complete_extinction())

print("n array shape:", n_data.shape)
print("First 5 rows (wavelength µm, n):")
print(n_data[:5])

print("\nk array shape:", k_data.shape)
print("First 5 rows (wavelength µm, k):")
print(k_data[:5])

The `Database` class also has direct convenience methods that load the material and return the array in one step:

In [ ]:
n_arr = db.get_material_n_numpy(5)
k_arr = db.get_material_k_numpy(5)
print("n array:\n", n_arr)
print("\nk array:\n", k_arr)

### 3.5 Query at a specific wavelength (default: nm)

Pass wavelength in **nanometres** (default). The value is interpolated if it falls within the measured range; otherwise **NaN** is returned.

In [ ]:
from refractivesqlite.exceptions import NoExtinctionCoefficient

wavelength_nm = 633  # He-Ne laser line

n_at_wl = mat.get_refractiveindex(wavelength_nm)
print(f"n at {wavelength_nm} nm = {n_at_wl:.4f}")

try:
    k_at_wl = mat.get_extinctioncoefficient(wavelength_nm)
    print(f"k at {wavelength_nm} nm = {k_at_wl:.4f}")
except NoExtinctionCoefficient:
    print("No extinction coefficient data for this material.")

### 3.6 Export to CSV

`to_csv` writes one or two files depending on what data is available:
- `filename(n).csv` — wavelength + n
- `filename(k).csv` — wavelength + k  
- `filename(nk).csv` — wavelength + n + k (when both are defined at the same points)

In [ ]:
mat.to_csv(output="Ag_Johnson.csv")

In [ ]:
# Shortcut: load and export in one call
db.get_material_csv(5, folder="csv_output")

In [ ]:
# Export every material in the database (can take a while)
# db.get_material_csv_all(outputfolder="all_csv")

---
## 4. Visualisation

The data returned by `get_complete_refractive()` and `get_complete_extinction()` is plain Python lists, so it plugs straight into matplotlib.

### 4.1 Plot *n* and *k* for a single material

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mat = db.get_material(5)   # Ag / Johnson
info = mat.get_page_info()

n_data = np.array(mat.get_complete_refractive())
k_data = np.array(mat.get_complete_extinction())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(n_data[:, 0], n_data[:, 1], color="steelblue", linewidth=2)
ax1.set_xlabel("Wavelength (µm)")
ax1.set_ylabel("Refractive index n")
ax1.set_title(f"{info['shelf']} / {info['book']} / {info['page']}")
ax1.grid(True, alpha=0.3)

ax2.plot(k_data[:, 0], k_data[:, 1], color="tomato", linewidth=2)
ax2.set_xlabel("Wavelength (µm)")
ax2.set_ylabel("Extinction coefficient k")
ax2.set_title(f"{info['shelf']} / {info['book']} / {info['page']}")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 Compare *n* across multiple materials

Use `search_custom` to find the page IDs you want, then loop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Find all gold (Au) pages that have refractive index data
rows = db.search_custom(
    'SELECT pageid, page FROM pages WHERE book="Au" AND hasrefractive=1'
)

fig, ax = plt.subplots(figsize=(9, 5))

for pageid, page_name in rows:
    mat = db.get_material(pageid)
    n_data = np.array(mat.get_complete_refractive())
    ax.plot(n_data[:, 0], n_data[:, 1], label=page_name, linewidth=1.5)

ax.set_xlabel("Wavelength (µm)")
ax.set_ylabel("Refractive index n")
ax.set_title("Gold (Au) — refractive index across datasets")
ax.set_xlim(0, 2)
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Advanced imports

For more control, import sub-modules directly.

### 5.1 Build a `Material` from your own data

`Material.FromLists` lets you wrap arbitrary *n* / *k* arrays — no database required.

In [ ]:
import numpy as np
from refractivesqlite import Material

# Synthetic Cauchy-like data
wavelengths = list(np.linspace(0.4, 0.9, 50))
n_values    = [1.5 + 0.01 / w**2 for w in wavelengths]
k_values    = [1e-4 / w**3       for w in wavelengths]

pageinfo = {"pageid": 0, "shelf": "custom", "book": "MyGlass", "page": "synthetic"}

mat = Material.FromLists(
    pageinfo,
    wavelengths_r=wavelengths, refractive=n_values,
    wavelengths_e=wavelengths, extinction=k_values,
)

print("Has n?", mat.has_refractive())
print("Has k?", mat.has_extinction())
print(f"n at 550 nm = {mat.get_refractiveindex(550):.5f}")
print(f"k at 550 nm = {mat.get_extinctioncoefficient(550):.2e}")

### 5.2 Handling exceptions

Specific exception types are in `refractivesqlite.exceptions`.

In [ ]:
from refractivesqlite.exceptions import NoExtinctionCoefficient
from refractivesqlite import Material

# Material with n only
pageinfo = {"pageid": 0, "shelf": "demo", "book": "Glass", "page": "n_only"}
mat_n = Material.FromLists(
    pageinfo,
    wavelengths_r=[0.4, 0.6, 0.8],
    refractive=[1.52, 1.50, 1.49],
)

try:
    mat_n.get_extinctioncoefficient(500)
except NoExtinctionCoefficient as e:
    print("Caught expected exception:", e)

### 5.3 Using the data models directly

`models.py` exposes the `Shelf`, `Book`, `Page`, and `Entry` namedtuples used internally by the builder.

In [ ]:
from refractivesqlite.models import Shelf, Book, Page, Entry

shelf = Shelf(shelf="main", name="Main optical materials")
book  = Book(book="SiO2", name="Silicon dioxide")
page  = Page(page="Malitson", name="Malitson 1965", path="/path/to/file.yml")

print(shelf)
print(book)
print(page)

### 5.4 Low-level builder access

You can call the builder functions directly to inspect the YML catalogue without going through a `Database` object.

> The builder caches the parsed `library.yml` in memory, so repeated calls to `extract_entry_list` with the same path incur no additional I/O.

In [ ]:
from refractivesqlite.builder import extract_entry_list, print_pretty_entry_list

# entries = extract_entry_list("database")   # path to your local YML folder
# print(f"Found {len(entries)} entries")
# print_pretty_entry_list(entries[:10])       # show first 10
print("(Uncomment the lines above when a local YML folder is available.)")

---
## 6. Multi-unit and vectorised queries

All query methods — `get_refractiveindex`, `get_extinctioncoefficient`, `get_epsilon`, and `get_wl_range` — accept an optional `unit=` keyword.  
Supported units are: **`'m'`, `'mm'`, `'um'`, `'nm'`** (default)**, `'A'`, `'cm-1'`, `'THz'`, `'eV'`**.

All methods also accept **scalar or array-like** wavelength input.  
Wavelengths outside the material's valid range return **NaN** instead of raising an exception.

### 6.1 Scalar queries in different units

The following four calls all query the same physical wavelength (≈ 633 nm).

In [ ]:
import numpy as np
from refractivesqlite import Material

# Build a simple glass with n data in um (as the database stores internally)
pageinfo = {"pageid": 0, "shelf": "demo", "book": "Glass", "page": "Cauchy"}
wl_um = list(np.linspace(0.3, 1.0, 100))
n_vals = [1.5 + 0.005 / w**2 for w in wl_um]
mat = Material.FromLists(pageinfo, wavelengths_r=wl_um, refractive=n_vals)

# All four evaluate at the same point
print(f"n(633 nm)       = {mat.get_refractiveindex(633, unit='nm'):.5f}")
print(f"n(0.633 um)     = {mat.get_refractiveindex(0.633, unit='um'):.5f}")
print(f"n(6330 A)       = {mat.get_refractiveindex(6330, unit='A'):.5f}")
print(f"n(15798 cm-1)   = {mat.get_refractiveindex(15798, unit='cm-1'):.5f}")
print(f"n(1.959 eV)     = {mat.get_refractiveindex(1.959, unit='eV'):.5f}")
print(f"n(473.6 THz)    = {mat.get_refractiveindex(473.6, unit='THz'):.5f}")

### 6.2 Vectorised query — array of wavelengths

Pass a NumPy array; the result is a matching NumPy array. Values outside the valid range are `NaN`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from refractivesqlite import Material

pageinfo = {"pageid": 0, "shelf": "demo", "book": "Glass", "page": "Cauchy"}
wl_um = list(np.linspace(0.3, 1.0, 100))
n_vals = [1.5 + 0.005 / w**2 for w in wl_um]
mat = Material.FromLists(pageinfo, wavelengths_r=wl_um, refractive=n_vals)

# Dense wavelength grid — some points outside the material range
wls = np.linspace(200, 1200, 500)          # nm
n   = mat.get_refractiveindex(wls)         # ndarray, NaN outside [300, 1000] nm

print(f"Query array shape : {n.shape}")
print(f"NaN count         : {np.sum(np.isnan(n))}  "
      f"(wavelengths outside [300, 1000] nm)")

plt.figure(figsize=(8, 3))
plt.plot(wls, n, linewidth=2, color='steelblue')
plt.xlabel("Wavelength (nm)")
plt.ylabel("n")
plt.title("Vectorised query — NaN outside valid range")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Query in eV — common in photonics and solid-state physics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from refractivesqlite import Material

pageinfo = {"pageid": 0, "shelf": "demo", "book": "Glass", "page": "Cauchy"}
wl_um = list(np.linspace(0.3, 1.0, 100))
n_vals = [1.5 + 0.005 / w**2 for w in wl_um]
mat = Material.FromLists(pageinfo, wavelengths_r=wl_um, refractive=n_vals)

energies = np.linspace(1.2, 4.5, 300)     # eV
n = mat.get_refractiveindex(energies, unit='eV')

plt.figure(figsize=(8, 3))
plt.plot(energies, n, linewidth=2, color='darkorange')
plt.xlabel("Photon energy (eV)")
plt.ylabel("n")
plt.title("Refractive index vs photon energy")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.4 Complex permittivity ε

`get_epsilon` returns ε = (n + ik)² using the **exp(−iωt)** physics convention by default.  
Pass `convention='exp_plus_i_omega_t'` for the engineering (EE) sign convention.  
Accepts the same `unit=` keyword and array input as the other query methods.

In [ ]:
import numpy as np
from refractivesqlite import Material

# Material with both n and k
pageinfo = {"pageid": 0, "shelf": "demo", "book": "Metal", "page": "synthetic"}
wl_um  = list(np.linspace(0.3, 1.0, 80))
n_vals = [0.2 + 0.1 * w for w in wl_um]
k_vals = [3.5 - 1.0 * w for w in wl_um]

mat = Material.FromLists(
    pageinfo,
    wavelengths_r=wl_um, refractive=n_vals,
    wavelengths_e=wl_um, extinction=k_vals,
)

eps = mat.get_epsilon(633)           # nm, physics convention
n   = mat.get_refractiveindex(633)
k   = mat.get_extinctioncoefficient(633)

print(f"n  = {n:.4f}")
print(f"k  = {k:.4f}")
print(f"ε  = {eps:.4f}")
print(f"ε  = ({eps.real:.4f}) + ({eps.imag:.4f})i")
print(f"\nVerification: (n+ik)² = ({n:.4f}+{k:.4f}i)² = {(n+1j*k)**2:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wls = np.linspace(300, 1000, 300)       # nm
eps = mat.get_epsilon(wls)              # shape (300,), complex

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(wls, eps.real, color='steelblue', linewidth=2)
ax1.axhline(0, color='k', linewidth=0.6, linestyle='--')
ax1.set_xlabel("Wavelength (nm)")
ax1.set_ylabel(r"Re(ε)")
ax1.set_title("Real part of permittivity")
ax1.grid(True, alpha=0.3)

ax2.plot(wls, eps.imag, color='tomato', linewidth=2)
ax2.set_xlabel("Wavelength (nm)")
ax2.set_ylabel(r"Im(ε)")
ax2.set_title("Imaginary part of permittivity")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.5 Wavelength range in any unit

`get_wl_range(unit)` returns the `(min, max)` valid wavelength range converted to the requested unit.

In [ ]:
import numpy as np
from refractivesqlite import Material

pageinfo = {"pageid": 0, "shelf": "demo", "book": "Glass", "page": "Cauchy"}
wl_um = list(np.linspace(0.4, 0.9, 50))
n_vals = [1.5 + 0.01 / w**2 for w in wl_um]
mat = Material.FromLists(pageinfo, wavelengths_r=wl_um, refractive=n_vals)

for unit in ['nm', 'um', 'm', 'A', 'cm-1', 'THz', 'eV']:
    lo, hi = mat.get_wl_range(unit=unit)
    print(f"  {unit:>5s} : ({lo:.4g}, {hi:.4g})")

### 6.6 Direct access to the unit registry

The `_units` module is private but importable if you need standalone wavelength conversions.

In [ ]:
from refractivesqlite._units import to_nm, from_nm, SUPPORTED_UNITS
import numpy as np

print("Supported units:", SUPPORTED_UNITS)

wl_eV = np.array([1.0, 2.0, 3.0, 4.0])   # eV
wl_nm = to_nm(wl_eV, 'eV')               # → nm
wl_back = from_nm(wl_nm, 'eV')            # → eV

print("\nInput (eV):", wl_eV)
print("→ nm:      ", wl_nm.round(2))
print("→ back eV: ", wl_back.round(6))